In [112]:
import numpy as np
import pandas as pd

In [113]:
bioinfo = pd.read_parquet('../data/raw/bioinfo.parquet')  # file path
ideology = pd.read_parquet('../data/raw/ideology.parquet')  # file path
terms = pd.read_parquet('../data/raw/terms.parquet')
bioinfo.info()

bill_summaries_119 = pd.read_parquet('../data/raw/bill_summaries_119.parquet')
sponsored_legislation = pd.read_parquet('../data/raw/sponsored_legislation.parquet')

committee_members = pd.read_parquet('../data/raw/committee_members.parquet')
committees = pd.read_parquet('../data/raw/committees.parquet')

schedule_e_2026 = pd.read_parquet('../data/raw/schedule_e_2026.parquet')
vote_compare = pd.read_parquet('../data/raw/vote_compare.parquet')

<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bioguide_id     555 non-null    str    
 1   Full name       555 non-null    str    
 2   Chamber         555 non-null    str    
 3   State           555 non-null    str    
 4   Party           555 non-null    str    
 5   District        439 non-null    float64
 6   birthYear       554 non-null    str    
 7   image           554 non-null    str    
 8   Office address  539 non-null    str    
 9   Phone           539 non-null    str    
 10  Website         539 non-null    str    
dtypes: float64(1), str(10)
memory usage: 166.9 KB


In [114]:
bioinfo['bioguide_id'].duplicated().value_counts()  # checking primary key

bioguide_id
False    555
Name: count, dtype: int64

In [115]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
Full name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
Chamber,House of Representatives,House of Representatives,House of Representatives
State,Alabama,Alabama,Alabama
Party,Republican,Democratic,Republican
District,3.0,7.0,6.0
birthYear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
Office address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
Phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [116]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.
    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.
    Returns:
        A DBML string representing the DataFrame schema.
    """
    dbml_string = f"Table {table_name} {{\n"
    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"
    dbml_string += "}\n"
    return dbml_string
def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [117]:
bioinfo.columns = [x.replace(' ', '_').lower() for x in bioinfo.columns]
print(pandas_df_to_dbml(bioinfo, 'members'))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
}



In [118]:
ideology.info()

<class 'pandas.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   bioname              557 non-null    str    
 1   chamber              557 non-null    str    
 2   left_right_ideology  555 non-null    float64
 3   state_abbrev         557 non-null    str    
 4   district_code        557 non-null    int64  
 5   icpsr                557 non-null    int64  
 6   bioguide_id          556 non-null    str    
 7   party                557 non-null    str    
 8   fec_id               538 non-null    object 
dtypes: float64(1), int64(2), object(1), str(5)
memory usage: 60.7+ KB


In [119]:
ideology.head(3).T

,0,1,2
bioname,"TRUMP, Donald John","ROGERS, Mike Dennis","SEWELL, Terri"
chamber,President,House,House
left_right_ideology,0.403,0.379,-0.402
state_abbrev,USA,AL,AL
district_code,0,3,7
icpsr,99912,20301,21102
bioguide_id,NaN,R000575,S001185
party,Republican,Republican,Democrat
fec_id,None,[H2AL03032],[H0AL07086]


In [120]:
ideology = ideology[['icpsr', 'bioguide_id', 'fec_id', 'left_right_ideology']]

In [121]:
ideology

,icpsr,bioguide_id,fec_id,left_right_ideology
0,99912,NaN,None,0.403
1,20301,R000575,[H2AL03032],0.379
2,21102,S001185,[H0AL07086],-0.402
3,21500,P000609,[H4AL06098],0.670
4,22140,M001212,"[H8AL02171, S6AL00476]",0.646
...,...,...,...,...
552,42504,J000312,[S4WV00332],0.562
553,29940,B001230,"[H8WI00018, S2WI00219]",-0.484
554,41111,J000293,[S0WI00197],0.648
555,20953,L000571,"[H8WY00148, S0WY00137]",0.682


In [122]:
fec = ideology.explode('fec_id')
ideology = ideology.drop(['fec_id'], axis=1)
fec = fec.drop(['bioguide_id', 'left_right_ideology'], axis=1)

In [123]:
ideology

,icpsr,bioguide_id,left_right_ideology
0,99912,NaN,0.403
1,20301,R000575,0.379
2,21102,S001185,-0.402
3,21500,P000609,0.670
4,22140,M001212,0.646
...,...,...,...
552,42504,J000312,0.562
553,29940,B001230,-0.484
554,41111,J000293,0.648
555,20953,L000571,0.682


In [124]:
fec

,icpsr,fec_id
0,99912,NaN
1,20301,H2AL03032
2,21102,H0AL07086
3,21500,H4AL06098
4,22140,H8AL02171
...,...,...
553,29940,S2WI00219
554,41111,S0WI00197
555,20953,H8WY00148
555,20953,S0WY00137


In [125]:
ideology = ideology.groupby(['bioguide_id']).agg({'left_right_ideology': 'mean'}).reset_index()

In [126]:
members = pd.merge(bioinfo, ideology, on = 'bioguide_id', how = 'outer', validate = 'one_to_one', indicator = 'matched')
members['matched'].value_counts()
members = members.drop(['matched'], axis=1)

In [127]:
ideology[ideology['bioguide_id'].duplicated()]

,bioguide_id,left_right_ideology


In [128]:
ideology.query("bioguide_id == 'K000401'")

,bioguide_id,left_right_ideology
277,K000401,0.2025


In [129]:
print(pandas_df_to_dbml(members, 'members'))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
  left_right_ideology float
}



In [130]:
fec

,icpsr,fec_id
0,99912,NaN
1,20301,H2AL03032
2,21102,H0AL07086
3,21500,H4AL06098
4,22140,H8AL02171
...,...,...
553,29940,S2WI00219
554,41111,S0WI00197
555,20953,H8WY00148
555,20953,S0WY00137


In [131]:
print(pandas_df_to_dbml(fec, 'crosswalk'))

Table crosswalk {
  icpsr int
  fec_id varchar
}



In [132]:
terms = terms.drop(['startYear', 'endYear'], axis=1)
terms.columns = [x.lower() for x in terms.columns]
print(pandas_df_to_dbml(terms, "terms"))

Table terms {
  bioguide_id varchar
  chamber varchar
  congress int
  statecode varchar
  district float
}



In [133]:
members.to_csv("../data/processed/members.csv", index=False)
fec.to_csv("../data/processed/crosswalk.csv", index=False)
terms.to_csv("../data/processed/terms.csv", index=False)

## Bill summaries and sponsored legislation

In [134]:
bill_summaries_119.head(3).T

,0,1,2
actionDate,2025-01-03,2025-01-09,2025-01-06
actionDesc,Introduced in House,Introduced in House,Introduced in Senate
currentChamber,House,House,Senate
currentChamberCode,H,H,S
lastSummaryUpdateDate,2025-01-09T01:58:40Z,2025-01-13T12:49:11Z,2025-01-13T16:21:37Z
text,<p><strong>Illegitimate Court Counteraction Ac...,<p>This resolution disapproves of the Central ...,<p><strong>Laken Riley Act</strong></p><p>This...
updateDate,2025-01-09T01:58:53Z,2025-01-13T12:51:00Z,2025-01-13T16:21:43Z
versionCode,00,00,00
bill.congress,119,119,119
bill.number,23,27,5


In [135]:
sponsored_legislation.head(3).T

,0,1,2
introducedDate,2026-05-13,2026-07-22,2026-07-22
number,8800,NaN,NaN
title,National Defense Authorization Act for Fiscal ...,NaN,NaN
type,HR,NaN,NaN
url,https://api.congress.gov/v3/bill/119/hr/8800?f...,https://api.congress.gov/v3/amendment/119/hamd...,https://api.congress.gov/v3/amendment/119/hamd...
bioguide_id,R000575,R000575,R000575


In [136]:
from bs4 import BeautifulSoup
mysoup = BeautifulSoup(bill_summaries_119['text'][0], 'html.parser')
mysoup.text

'Illegitimate Court Counteraction ActThis bill imposes sanctions against foreign persons (individuals and entities) who assist the International Criminal Court (ICC) in investigating, arresting, detaining, or prosecuting certain individuals.The bill categorizes as protected persons (1) any U.S.\xa0individual, U.S. entity, or person in the United States, unless the United States is a state party to the Rome Statute of the ICC and provides formal consent to ICC jurisdiction; and (2) any foreign person that is a citizen or lawful resident of a U.S. ally that is not a state party to the Rome Statute or has not consented to ICC jurisdiction.If the ICC attempts to investigate, arrest, detain or prosecute a protected person, the President must impose visa- and property-blocking sanctions against the foreign persons that engaged in or materially assisted in such actions, as well as against foreign persons owned by, controlled by, or acting on behalf of such foreign persons. The President must 

In [137]:
bill_summaries_119.shape

(6146, 16)

In [138]:
sponsored_legislation.shape

(26610, 6)

In [139]:
sponsored_legislation['url'].duplicated().value_counts()

url
False    26600
True        10
Name: count, dtype: int64

In [140]:
sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]

,introducedDate,number,title,type,url,bioguide_id
13937,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
13938,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
15083,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
15085,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
20372,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20373,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20374,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20375,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20379,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20380,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608


In [141]:
[print(x) for x in sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]]

introducedDate
number
title
type
url
bioguide_id


[None, None, None, None, None, None]

In [142]:
bill_summaries_119[bill_summaries_119['bill.url'].duplicated(keep=False)]

,actionDate,actionDesc,currentChamber,currentChamberCode,lastSummaryUpdateDate,text,updateDate,versionCode,bill.congress,bill.number,bill.originChamber,bill.originChamberCode,bill.title,bill.type,bill.updateDateIncludingText,bill.url
2,2025-01-06,Introduced in Senate,Senate,S,2025-01-13T16:21:37Z,<p><strong>Laken Riley Act</strong></p><p>This...,2025-01-13T16:21:43Z,00,119,5,Senate,S,Laken Riley Act,S,2026-07-30,https://api.congress.gov/v3/bill/119/s/5?forma...
44,2025-01-16,Introduced in House,House,H,2025-01-27T11:42:14Z,<p>This joint resolution nullifies the rule ti...,2025-01-27T11:44:23Z,00,119,24,House,H,Providing for congressional disapproval under ...,HJRES,2026-07-13,https://api.congress.gov/v3/bill/119/hjres/24?...
45,2025-01-15,Introduced in House,House,H,2025-01-27T14:02:08Z,<p>This joint resolution nullifies the rule ti...,2025-01-27T14:02:43Z,00,119,20,House,H,Providing for congressional disapproval under ...,HJRES,2026-09-19,https://api.congress.gov/v3/bill/119/hjres/20?...
54,2025-01-03,Introduced in House,House,H,2025-01-30T16:54:11Z,<p><b>Wounded Knee Massacre Memorial and Sacre...,2025-01-30T16:54:15Z,00,119,165,House,H,Wounded Knee Massacre Memorial and Sacred Site...,HR,2026-07-13,https://api.congress.gov/v3/bill/119/hr/165?fo...
80,2025-01-03,Introduced in House,House,H,2025-02-03T14:54:09Z,<p><strong>Alaska Native Settlement Trust Elig...,2025-02-03T14:56:55Z,00,119,42,House,H,Alaska Native Settlement Trust Eligibility Act,HR,2026-07-13,https://api.congress.gov/v3/bill/119/hr/42?for...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5815,2026-09-16,Public Law,Unknown,Unknown,2026-09-18T19:00:10Z,<p><strong>Prison Staff Safety Enhancement Act...,2026-09-18T19:00:49Z,49,119,307,Senate,S,Prison Staff Safety Enhancement Act,S,2026-09-21,https://api.congress.gov/v3/bill/119/s/307?for...
5816,2026-09-11,Public Law,Unknown,Unknown,2026-09-18T19:07:13Z,<p><strong>Doug LaMalfa Federal Disaster Tax R...,2026-09-18T19:12:06Z,49,119,5366,House,H,Doug LaMalfa Federal Disaster Tax Relief Certa...,HR,2026-09-21,https://api.congress.gov/v3/bill/119/hr/5366?f...
5823,2026-06-24,Reported to Senate,Senate,S,2026-09-21T20:53:33Z,<p><strong>Protect College Sports Act of 2026<...,2026-09-21T20:54:12Z,25,119,4668,Senate,S,Protect College Sports Act of 2026,S,2026-10-07,https://api.congress.gov/v3/bill/119/s/4668?fo...
5917,2026-09-15,Passed House,House,H,2026-09-28T14:52:37Z,<p><strong>Protecting Taxpayers from Ghost Pre...,2026-09-28T14:57:21Z,53,119,9499,House,H,Protecting Taxpayers from Ghost Preparers Act,HR,2026-09-28,https://api.congress.gov/v3/bill/119/hr/9499?f...


In [143]:
sponsored_legislation = sponsored_legislation.drop_duplicates()
sponsored_legislation = sponsored_legislation[['bioguide_id', 'url']]
bills = pd.merge(bill_summaries_119, sponsored_legislation, left_on = ['bill.url'], right_on=['url'], how = 'outer', indicator = 'matched', validate = 'many_to_one')

In [144]:
bills['matched'].value_counts()

matched
right_only    20597
both           6146
left_only         0
Name: count, dtype: int64

In [145]:
bills = bills.query("matched == 'both'")
bills = bills.drop('matched', axis=1)
bills.columns = [x.lower().replace('.', '_') for x in bills.columns]
bills

,actiondate,actiondesc,currentchamber,currentchambercode,lastsummaryupdatedate,text,updatedate,versioncode,bill_congress,bill_number,bill_originchamber,bill_originchambercode,bill_title,bill_type,bill_updatedateincludingtext,bill_url,bioguide_id,url
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...


In [146]:
bills.duplicated(['bill_url', 'versioncode']).value_counts()

False    6146
Name: count, dtype: int64

In [147]:
bills = bills.drop('url', axis=1)
bills.to_csv("../data/processed/bills.csv", index=False)

In [148]:
print(pandas_df_to_dbml(bills, 'bills'))

Table bills {
  actiondate varchar
  actiondesc varchar
  currentchamber varchar
  currentchambercode varchar
  lastsummaryupdatedate varchar
  text varchar
  updatedate varchar
  versioncode varchar
  bill_congress float
  bill_number varchar
  bill_originchamber varchar
  bill_originchambercode varchar
  bill_title varchar
  bill_type varchar
  bill_updatedateincludingtext varchar
  bill_url varchar
  bioguide_id varchar
}



## FEC Schedule E

In [149]:
pd.set_option('display.max_rows', 128)
schedule_e_2026.head(3).T

,0,1,2
fec_id,H4AL06098,H4AL06098,H8AL02171
bioguide_id,P000609,P000609,M001212
action_code,A,A,A
action_code_full,ADD,ADD,ADD
amendment_indicator,N,N,N
amendment_number,0.0,NaN,0.0
back_reference_schedule_name,NaN,NaN,NaN
back_reference_transaction_id,NaN,NaN,NaN
candidate_first_name,GARY,GARY,FELIX
candidate_id,H4AL06098,H4AL06098,H8AL02171


In [152]:
schedule_e_2026.columns

# Who was the expenditure made for? fec_id 

# How much was the expenditure? expenditure_amount

# Who made the expenditure? committee_name

# support_oppose_indicator committee-committee_type_full

# Super PAC or independent expenditure?

expenditures = schedule_e_2026[['fec_id', 'transaction_id', 'expenditure_amount', 'committee.name', 'committee.committee_type_full', 'support_oppose_indicator']]
expenditures = expenditures.rename({'committee.name': 'committee_name', 'committee.committee_type_full': 'committee_type_full'}, axis=1)

In [153]:
print(pandas_df_to_dbml(expenditures, "expenditures"))

Table expenditures {
  fec_id varchar
  transaction_id varchar
  expenditure_amount float
  committee_name varchar
  committee_type_full varchar
  support_oppose_indicator varchar
}

